In [5]:
import pandas as pd
import os
from datetime import datetime, timedelta
directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2017/'

In [6]:
# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

monthly_data = []

year = 2017

for month in range(1, 13):  # Loop monthly
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:  # Match files by month and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
            # Load message and orderbook 
            message_df = pd.read_csv(message_file)
            orderbook_df = pd.read_csv(orderbook_file)

            # Message and orderbook data, dropper col 7
            message_df = message_df.iloc[:, :-1]  
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            base_date = filename.split('_')[1]  # Extract base date from filename (assumes 'SPY_2017-01-03_...')
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merger message and orderbook data on 'Time (sec)'
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Drop NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' index for 1 second
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            # list monthly
            monthly_files.append(resampled_df)
    
    # Concatenate all daily filer for month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        # Gem
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)

# All months into one final DataFrame
final_df = pd.concat(monthly_data)

# Save the final DataFrame, year 2017
final_df.to_csv(f'combined_SPY{year}_cleaned.csv', index=False)


print(final_df.head())


<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)
<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)
<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)
<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)
<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  message_df = pd.read_csv(message_file)
<ipython-input-6-02bd2723ba87>:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2017-01-03 09:30:00    2250500.0       100.0    2250400.0       499.0   
2017-01-03 09:30:01    2251000.0       200.0    2250800.0      5451.0   
2017-01-03 09:30:02    2251000.0       300.0    2250800.0       900.0   
2017-01-03 09:30:03    2251100.0       400.0    2250900.0      1600.0   
2017-01-03 09:30:04    2251000.0      4100.0    2250800.0       300.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2017-01-03 09:30:00    2250600.0      1700.0    2250300.0      4700.0   
2017-01-03 09:30:01    2251100.0       200.0    2250700.0      8500.0   
2017-01-03 09:30:02    2251100.0      1300.0    2250700.0      6300.0   
2017-01-03 09:30:03    2251200.0      3300.0    2250800.0       400.0   
2017-01-03 09:30:04    2251100.0      1900.0    22

In [9]:
print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2017-01-03 09:30:00    2250500.0       100.0    2250400.0       499.0   
2017-01-03 09:30:01    2251000.0       200.0    2250800.0      5451.0   
2017-01-03 09:30:02    2251000.0       300.0    2250800.0       900.0   
2017-01-03 09:30:03    2251100.0       400.0    2250900.0      1600.0   
2017-01-03 09:30:04    2251000.0      4100.0    2250800.0       300.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2017-01-03 09:30:00    2250600.0      1700.0    2250300.0      4700.0   
2017-01-03 09:30:01    2251100.0       200.0    2250700.0      8500.0   
2017-01-03 09:30:02    2251100.0      1300.0    2250700.0      6300.0   
2017-01-03 09:30:03    2251200.0      3300.0    2250800.0       400.0   
2017-01-03 09:30:04    2251100.0      1900.0    22

In [10]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2017-12-29 15:59:55    2666500.0      9195.0    2666400.0      1100.0   
2017-12-29 15:59:56    2667000.0      7400.0    2666900.0      2000.0   
2017-12-29 15:59:57    2667000.0      8400.0    2666800.0      5100.0   
2017-12-29 15:59:58    2667300.0      3300.0    2667200.0      2600.0   
2017-12-29 15:59:59    2668200.0      5200.0    2668100.0      2200.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2017-12-29 15:59:55    2666600.0     10400.0    2666300.0      6200.0   
2017-12-29 15:59:56    2667100.0     11100.0    2666800.0      6900.0   
2017-12-29 15:59:57    2667100.0      7600.0    2666700.0      9100.0   
2017-12-29 15:59:58    2667400.0     10800.0    2667100.0      4253.0   
2017-12-29 15:59:59    2668300.0      9500.0    26

In [11]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5873218


In [12]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2017-01-03 09:30:00,2250500.0,100.0,2250400.0,499.0,2250600.0,1700.0,2250300.0,4700.0,1.0,7955180.0,200.0,2250400.0,1.0
2017-01-03 09:30:01,2251000.0,200.0,2250800.0,5451.0,2251100.0,200.0,2250700.0,8500.0,1.0,8102992.0,200.0,2251000.0,-1.0
2017-01-03 09:30:02,2251000.0,300.0,2250800.0,900.0,2251100.0,1300.0,2250700.0,6300.0,3.0,8239508.0,2000.0,2250700.0,1.0
2017-01-03 09:30:03,2251100.0,400.0,2250900.0,1600.0,2251200.0,3300.0,2250800.0,400.0,1.0,8443204.0,200.0,2251100.0,-1.0
2017-01-03 09:30:04,2251000.0,4100.0,2250800.0,300.0,2251100.0,1900.0,2250700.0,1700.0,4.0,8545096.0,100.0,2251000.0,-1.0


In [13]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2017-12-29 15:59:55,2666500.0,9195.0,2666400.0,1100.0,2666600.0,10400.0,2666300.0,6200.0,3.0,147659260.0,1000.0,2666400.0,1.0
2017-12-29 15:59:56,2667000.0,7400.0,2666900.0,2000.0,2667100.0,11100.0,2666800.0,6900.0,1.0,147753616.0,1000.0,2667000.0,-1.0
2017-12-29 15:59:57,2667000.0,8400.0,2666800.0,5100.0,2667100.0,7600.0,2666700.0,9100.0,3.0,147801308.0,500.0,2666800.0,1.0
2017-12-29 15:59:58,2667300.0,3300.0,2667200.0,2600.0,2667400.0,10800.0,2667100.0,4253.0,3.0,147867180.0,1000.0,2667300.0,-1.0
2017-12-29 15:59:59,2668200.0,5200.0,2668100.0,2200.0,2668300.0,9500.0,2668000.0,12300.0,1.0,147978340.0,1000.0,2668200.0,-1.0


In [15]:
# Save 
final_df.to_csv('final_combined_2017.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2017_with_time.csv', index=True)

In [16]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()


print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 251


In [17]:
# Load the data from 'final_combined_2017_with_time.csv'
final_combined_2017_with_time = pd.read_csv('final_combined_2017_with_time.csv')


final_combined_2017_with_time['Time (sec)'] = pd.to_datetime(final_combined_2017_with_time['Time (sec)'])

# Group between trading days
grouped = final_combined_2017_with_time.groupby(final_combined_2017_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate each 
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
     
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index Time (sec)'
resampled_5min_final_df.reset_index(inplace=True)

In [27]:
print(resampled_5min_final_df.head(10))

# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2017_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2017-01-03 09:30:00          2250500.0        2251500.0        2248400.0   
1 2017-01-03 09:35:00          2251000.0        2252700.0        2248100.0   
2 2017-01-03 09:40:00          2248700.0        2250000.0        2247900.0   
3 2017-01-03 09:45:00          2249600.0        2250600.0        2247700.0   
4 2017-01-03 09:50:00          2248600.0        2248900.0        2247300.0   
5 2017-01-03 09:55:00          2248600.0        2250000.0        2247900.0   
6 2017-01-03 10:00:00          2250100.0        2251700.0        2249200.0   
7 2017-01-03 10:05:00          2251600.0        2258300.0        2251100.0   
8 2017-01-03 10:10:00          2257400.0        2257700.0        2255400.0   
9 2017-01-03 10:15:00          2255700.0        2256400.0        2254800.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2251100.0          2250400.0        2251400.0        22

In [23]:
# Count observations for each day in 5-minute interval 
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2017-01-03    78
2017-01-04    78
2017-01-05    78
2017-01-06    78
2017-01-09    78
              ..
2017-12-22    78
2017-12-26    78
2017-12-27    78
2017-12-28    78
2017-12-29    78
Length: 251, dtype: int64


In [26]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2017_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2017-01-03 09:30:00          2250500.0        2251500.0        2248400.0   
1 2017-01-03 09:35:00          2251000.0        2252700.0        2248100.0   
2 2017-01-03 09:40:00          2248700.0        2250000.0        2247900.0   
3 2017-01-03 09:45:00          2249600.0        2250600.0        2247700.0   
4 2017-01-03 09:50:00          2248600.0        2248900.0        2247300.0   
5 2017-01-03 09:55:00          2248600.0        2250000.0        2247900.0   
6 2017-01-03 10:00:00          2250100.0        2251700.0        2249200.0   
7 2017-01-03 10:05:00          2251600.0        2258300.0        2251100.0   
8 2017-01-03 10:10:00          2257400.0        2257700.0        2255400.0   
9 2017-01-03 10:15:00          2255700.0        2256400.0        2254800.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2251100.0          2250400.0        2251400.0        22

In [25]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem her
resampled_5min_final_df.to_csv('resampled_5min_final_2017_corrected.csv', index=False)


Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2017-01-03 09:30:00,2250500.0,2251500.0,2248400.0,2251100.0,2250400.0,2251400.0,2248300.0,2251000.0,2250600.0,2251600.0,2248500.0,2251200.0,2250300.0,2251300.0,2248200.0,2250900.0,914941.0,3049.803333,419163.0,1397.210000,2119742.0,7065.806667,1100223.0,3667.410000,2250400.0,2251500.0,2248300.0,2250900.0,0.106667,2017-01-03
2017-01-03 09:35:00,2251000.0,2252700.0,2248100.0,2248700.0,2250800.0,2252600.0,2247900.0,2248600.0,2251100.0,2252800.0,2248200.0,2248800.0,2250700.0,2252500.0,2247800.0,2248500.0,637476.0,2124.920000,409199.0,1363.996667,1506614.0,5022.046667,1257792.0,4192.640000,2251000.0,2252700.0,2248100.0,2248500.0,0.333333,2017-01-03
2017-01-03 09:40:00,2248700.0,2250000.0,2247900.0,2249600.0,2248600.0,2249800.0,2247800.0,2249400.0,2248800.0,2250100.0,2248000.0,2249700.0,2248500.0,2249700.0,2247700.0,2249300.0,543912.0,1813.040000,436331.0,1454.436667,1230316.0,4101.053333,900172.0,3000.573333,2248700.0,2250000.0,2247800.0,2249300.0,0.186667,2017-01-03
2017-01-03 09:45:00,2249600.0,2250600.0,2247700.0,2248700.0,2249500.0,2250400.0,2247500.0,2248500.0,2249700.0,2250700.0,2247800.0,2248800.0,2249400.0,2250300.0,2247400.0,2248400.0,591041.0,1970.136667,708652.0,2362.173333,1418703.0,4729.010000,1423959.0,4746.530000,2249600.0,2250600.0,2247500.0,2248700.0,-0.086667,2017-01-03
2017-01-03 09:50:00,2248600.0,2248900.0,2247300.0,2248600.0,2248500.0,2248800.0,2247200.0,2248500.0,2248700.0,2249000.0,2247400.0,2248700.0,2248400.0,2248700.0,2247100.0,2248400.0,626461.0,2088.203333,878934.0,2929.780000,1454697.0,4848.990000,1650700.0,5502.333333,2248600.0,2249000.0,2247200.0,2248500.0,0.020000,2017-01-03
2017-01-03 09:55:00,2248600.0,2250000.0,2247900.0,2249800.0,2248500.0,2249800.0,2247800.0,2249600.0,2248700.0,2250100.0,2248000.0,2249900.0,2248400.0,2249700.0,2247700.0,2249500.0,439924.0,1466.413333,1259662.0,4198.873333,1027346.0,3424.486667,1750527.0,5835.090000,2248600.0,2249900.0,2247800.0,2249700.0,0.113333,2017-01-03
2017-01-03 10:00:00,2250100.0,2251700.0,2249200.0,2251600.0,2250000.0,2251600.0,2249100.0,2251500.0,2250200.0,2251800.0,2249300.0,2251700.0,2249900.0,2251500.0,2249000.0,2251400.0,523372.0,1744.573333,714106.0,2380.353333,1249213.0,4164.043333,1686204.0,5620.680000,2250100.0,2251700.0,2249200.0,2251700.0,-0.053333,2017-01-03
2017-01-03 10:05:00,2251600.0,2258300.0,2251100.0,2257300.0,2251400.0,2258200.0,2251000.0,2257200.0,2251700.0,2258400.0,2251200.0,2257400.0,2251300.0,2258100.0,2250900.0,2257100.0,465161.0,1550.536667,895620.0,2985.400000,1201860.0,4006.200000,1614630.0,5382.100000,2251300.0,2258300.0,2251000.0,2257400.0,0.000000,2017-01-03
2017-01-03 10:10:00,2257400.0,2257700.0,2255400.0,2255700.0,2257300.0,2257600.0,2255300.0,2255600.0,2257500.0,2257800.0,2255500.0,2255800.0,2257200.0,2257500.0,2255200.0,2255500.0,565854.0,1905.232323,995556.0,3352.040404,1174681.0,3955.154882,1452239.0,4889.693603,2257500.0,2257800.0,2255300.0,2255600.0,-0.010101,2017-01-03
2017-01-03 10:15:00,2255700.0,2256400.0,2254800.0,2255700.0,2255600.0,2256300.0,2254700.0,2255600.0,2255800.0,2256500.0,2254900.0,2255800.0,2255500.0,2256200.0,2254600.0,2255500.0,483222.0,1616.127090,1023177.0,3421.996656,1387908.0,4641.832776,1441643.0,4821.548495,2255600.0,2256400.0,2254700.0,2255700.0,-0.043478,2017-01-03
